In [0]:
query = """
SELECT 
    ROW_NUMBER() OVER(ORDER BY cc.customer_id) AS customer_key,
    cc.customer_id,
    cc.customer_number, 
    cc.first_name, 
    cc.last_name, 
    ecl.country,
    cc.marital_status, 
    CASE
        WHEN cc.gender <> "unknown" THEN cc.gender
        ELSE COALESCE (ec.gender, "unknown")
    END AS gender,
    ec.birth_date,
    cc.created_date AS create_date
FROM
    silver.crm_customers as cc
LEFT JOIN silver.erp_customers as ec
    ON cc.customer_number = ec.customer_number
LEFT JOIN silver.erp_customer_location as ecl
    ON cc.customer_number = ecl.customer_number
"""

In [0]:
df = spark.sql(query.replace('ON cc.customer_id = ecl.customer_id', 'ON cc.customer_number = ecl.customer_number'))

In [0]:
df.limit(5).display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("gold.dim_customers")

In [0]:
%sql
-- SELECT * FROM workspace.gold.gold_dim_customers LIMIT 5